In [4]:
# Run this first — install any missing libraries
!pip install -q xgboost shap imbalanced-learn kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix)
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings('ignore')

In [5]:
import kagglehub
import os
import pandas as pd

# Download dataset
path = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Path to dataset files:", path)

# Find the CSV file recursively (since it might be in a subdirectory)
csv_file = None
for root, dirs, files in os.walk(path):
    for file in files:
        if file == "accepted_2007_to_2018Q4.csv":
            csv_file = os.path.join(root, file)
            break
    if csv_file:
        break

if not csv_file:
    # If specific file not found, look for any accepted*.csv
    for root, dirs, files in os.walk(path):
        for file in files:
            if "accepted" in file and file.endswith('.csv'):
                csv_file = os.path.join(root, file)
                print(f"Found: {csv_file}")
                break
        if csv_file:
            break

print(f"Loading: {csv_file}")
df = pd.read_csv(csv_file, nrows=50000, low_memory=False)

# Keep key columns
cols = ['loan_status','loan_amnt','int_rate','annual_inc', 'dti','fico_range_low','grade','emp_length','home_ownership']
df = df[cols].copy()

# Binary target
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)
df.drop('loan_status', axis=1, inplace=True)

# Clean int_rate
df['int_rate'] = df['int_rate'].astype(str).str.replace('%','').astype(float)

# Encode emp_length
emp_map = {'< 1 year':0,'1 year':1,'2 years':2,'3 years':3,'4 years':4, '5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10}
df['emp_length'] = df['emp_length'].map(emp_map)

# Encode grade: A=0, B=1, ..., G=6
df['grade'] = df['grade'].map({'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6})

# One-hot encode home_ownership
df = pd.get_dummies(df, columns=['home_ownership'], drop_first=True)

# Drop rows with nulls
df.dropna(inplace=True)

print('Shape:', df.shape)
print('Default rate:', round(df.default.mean()*100, 2), '%')

Using Colab cache for faster access to the 'lending-club' dataset.
Path to dataset files: /kaggle/input/lending-club
Loading: /kaggle/input/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv
Shape: (47011, 11)
Default rate: 17.77 %


In [6]:
X = df.drop('default', axis=1)
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Training set after SMOTE:', X_train_bal.shape)
print('Class balance:', y_train_bal.value_counts().to_dict())

Training set after SMOTE: (61848, 10)
Class balance: {0: 30924, 1: 30924}


In [7]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model':     name,
        'accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'precision': round(precision_score(y_test, y_pred) * 100, 2),
        'recall':    round(recall_score(y_test, y_pred) * 100, 2),
        'f1':        round(f1_score(y_test, y_pred) * 100, 2),
        'auc':       round(roc_auc_score(y_test, y_prob) * 100, 2)
    }
    print(f'{name}: AUC={results[name]["auc"]}% Acc={results[name]["accuracy"]}%')

Logistic Regression: AUC=62.54% Acc=70.89%
Decision Tree: AUC=66.99% Acc=73.86%
Random Forest: AUC=64.47% Acc=80.02%
XGBoost: AUC=66.42% Acc=81.46%


In [8]:
roc_data = {}

for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    # Downsample to 50 points to keep JSON small
    step = max(1, len(fpr) // 50)
    roc_data[name] = {
        'fpr': fpr[::step].tolist(),
        'tpr': tpr[::step].tolist()
    }

In [9]:
rf_model = models['Random Forest']
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_
feat_imp = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:10]

feature_importance_data = [
    {'feature': f, 'importance': round(float(i)*100, 2)}
    for f, i in feat_imp
]
print(feature_importance_data)

[{'feature': 'int_rate', 'importance': 23.52}, {'feature': 'emp_length', 'importance': 12.11}, {'feature': 'dti', 'importance': 11.12}, {'feature': 'home_ownership_MORTGAGE', 'importance': 9.99}, {'feature': 'fico_range_low', 'importance': 9.96}, {'feature': 'annual_inc', 'importance': 9.92}, {'feature': 'loan_amnt', 'importance': 9.12}, {'feature': 'home_ownership_RENT', 'importance': 5.75}, {'feature': 'grade', 'importance': 4.64}, {'feature': 'home_ownership_OWN', 'importance': 3.89}]


In [10]:
import json

# Model metrics
with open('/content/model_metrics.json', 'w') as f:
    json.dump(list(results.values()), f)

# ROC curve data
with open('/content/roc_data.json', 'w') as f:
    json.dump(roc_data, f)

# Feature importance
with open('/content/feature_importance.json', 'w') as f:
    json.dump(feature_importance_data, f)

print('Done! Download these files from the Files panel:')
print('  model_metrics.json')
print('  roc_data.json')
print('  feature_importance.json')

Done! Download these files from the Files panel:
  model_metrics.json
  roc_data.json
  feature_importance.json
